In [1]:
from collections import defaultdict
from collections import Counter
from tqdm import tqdm
import pandas as pd
import numpy as np
import os

In [2]:
root = "/home/acomajuncosa/Documents_GPU/mtb-targeted-protein-degradation"

In [3]:
def get_splits(path):
    splits = os.listdir(path)
    return [i.replace(".csv", "") for i in sorted(splits)]

def get_pockets():
    df = pd.read_csv("/home/acomajuncosa/Documents_GPU/mtb-targeted-protein-degradation/processed/pocket_detection_data.csv")
    pockets = [f"{i.replace('.pdb', '')}_pocket_{j}" for i, j in zip(df['File name'], df['Pocket number'])]
    return sorted(pockets)

In [4]:
# Get splits (994)
SPLITS = get_splits(os.path.join(root, "processed", "unidock_REAL_docking", "inference_10B", 'A_proteins'))

# Get pockets (276)
POCKETS = get_pockets()

In [5]:
A_pockets = pd.read_csv(os.path.join(root, "processed", "unidock_REAL_docking", "inference_10B", "A_pockets.csv"))
B_pockets = pd.read_csv(os.path.join(root, "processed", "unidock_REAL_docking", "inference_10B", "B_pockets.csv"))
A_proteins = pd.read_csv(os.path.join(root, "processed", "unidock_REAL_docking", "inference_10B", "A_proteins.csv"))
B_proteins = pd.read_csv(os.path.join(root, "processed", "unidock_REAL_docking", "inference_10B", "B_proteins.csv"))

# Check that all pockets have same number of cpds associated
assert set(Counter(B_pockets['pocket']).values()) == set([1000])
assert set(Counter(B_proteins['pocket']).values()) == set([13000])

# Drop pocket column
B_pockets = B_pockets.drop(columns=['pocket'])
B_proteins = B_proteins.drop(columns=['pocket'])

# Include label
A_pockets['label'] = "A_pockets" 
B_pockets['label'] = "B_pockets" 
A_proteins['label'] = "A_proteins" 
B_proteins['label'] = "B_proteins"

# Merge all, sort and drop duplicates
COMPOUNDS = pd.concat([A_pockets, B_pockets, A_proteins, B_proteins]).sort_values(by=['split', 'index']).drop_duplicates(subset=['split', 'index']).reset_index(drop=True)

In [7]:
print(len(A_pockets) + len(B_pockets) + len(A_proteins) + len(B_proteins))
print(len(COMPOUNDS))
print((len(A_pockets) + len(B_pockets) + len(A_proteins) + len(B_proteins)) - len(COMPOUNDS))

1049000
1032646
16354


In [8]:
PATH_TO_SPLITS = os.path.join(root, "tmp")
PATH_TO_OUTPUT = os.path.join(root, "processed", "unidock_REAL_docking", "inference_10B", "selected_compounds")
os.makedirs(PATH_TO_OUTPUT, exist_ok=True)

In [9]:
ANNOTATED_COMPOUNDS = []

for split in tqdm(SPLITS[:100]):

    # Identify subset
    df = COMPOUNDS[COMPOUNDS['split'] == split].reset_index(drop=True)
    inds = df['index'].to_numpy()

    # Load split info
    smiles_ids = pd.read_csv(os.path.join(PATH_TO_SPLITS, f"{split}_SMILES_IDs.tsv.zip"), sep='\t')
    smiles_ids = smiles_ids.iloc[inds].reset_index()
    assert (smiles_ids['index'] == df['index']).all

    # Concatanate
    df = pd.concat([df, smiles_ids], axis=1)

    # Save
    df.to_csv(os.path.join(PATH_TO_OUTPUT, f"{split}.csv"), index=False)

100%|██████████| 100/100 [10:48<00:00,  6.48s/it]


In [10]:
def get_splits(path):
    splits = os.listdir(path)
    return [i.replace(".csv", "") for i in sorted(splits)]

def get_pockets():
    df = pd.read_csv("/home/acomajuncosa/Documents_GPU/mtb-targeted-protein-degradation/processed/pocket_detection_data.csv")
    pockets = [f"{i.replace('.pdb', '')}_pocket_{j}" for i, j in zip(df['File name'], df['Pocket number'])]
    return sorted(pockets)


In [11]:
PATH_TO_SELECTED_COMPOUNDS = os.path.join(root, "processed", "unidock_REAL_docking", "inference_10B", "selected_compounds")

In [41]:
df = []
for split in SPLITS:
    try:
        df_ = pd.read_csv(os.path.join(os.path.join(PATH_TO_OUTPUT, f"{split}.csv")))
        df.append(df_)
    except:
        pass
df = pd.concat(df, ignore_index=True)
rng = np.random.default_rng(42)
df["rd"] = rng.random(len(df))

CUSTOM_ORDER = ['A_proteins', 'A_pockets', 'B_proteins', 'B_pockets']
df["label"] = pd.Categorical(df["label"], categories=CUSTOM_ORDER, ordered=True)
df = df.sort_values(["label", "rd"], ascending=[True, True], kind="stable").reset_index(drop=True)

In [59]:
MAX_SYNTHON = 10
SYNTHON_COUNTS = defaultdict(int)
KEEP = []

for id_ in tqdm(df['id']):

    if id_.startswith("m_") == False and id_.startswith("s_") == False:
        raise TypeError("ID not starting with m nor s. Please revise")
    
    
    synthons = id_.split("____")[1:]
    keep = True
    tmp_dict = defaultdict(int)

    # tmp dict with synthons
    for synthon in synthons:
        tmp_dict[synthon] += 1

    # check valid compound
    for synthon in synthons:
        if SYNTHON_COUNTS[synthon] + tmp_dict[synthon] <= MAX_SYNTHON:
            pass
        else:
            keep = False
            break

    # add compound if valid
    if keep == True:
        KEEP.append(True)
        for synthon in synthons:
            SYNTHON_COUNTS[synthon] += 1
        
    else:
        KEEP.append(False)
        

df['keep'] = KEEP
d = Counter(KEEP)
print(round(100 * d[True] / (d[True] + d[False]), 2), "%")

100%|██████████| 113989/113989 [00:00<00:00, 906821.71it/s]

72.45 %


In [60]:
df

,split,index,n_targets,label,index.1,smiles,id,rd,keep
0,Enamine_REAL_LeadLike_008,4051326,21.0,A_proteins,4051326,FC1=C(C2=NC(C3CNCCO3)=NO2)SC=C1,m_271362____30583064____18521434,0.000013,True
1,Enamine_REAL_LeadLike_087,6127269,21.0,A_proteins,6127269,CC1=CNC(C(=O)NC2CN(C(=O)C3CNC(=O)N3)C2)=C1,m_275592____17554698____14140274____19055260,0.000056,True
2,Enamine_REAL_LeadLike_046,9127720,21.0,A_proteins,9127720,O=C(NCC1CSC1)C(=O)N1CCCC2=CC=CC=C21,s_2718____29034160____1019008,0.000057,True
3,Enamine_REAL_LeadLike_076,1487545,21.0,A_proteins,1487545,O=C(NC1=CN2N=CC=C2N=C1)C1=CC=C2NCCC2=C1,s_240690____13618256____9018314,0.000107,True
4,Enamine_REAL_LeadLike_027,7326378,21.0,A_proteins,7326378,CC1=NC=CC=C1NC(=O)C1=CC2=C(CNCC2)N1,s_240690____7399710____20276634,0.000152,True
...,...,...,...,...,...,...,...,...,...
113984,Enamine_REAL_LeadLike_044,9313705,50.0,B_pockets,9313705,C=C1CC2(C1)CN(C(=O)NC1=CN=C3NN=CC3=C1)C2,s_2430____15475208____26850320,0.999891,True
113985,Enamine_REAL_LeadLike_021,632022,50.0,B_pockets,632022,O=C(NNC1=CC(I)=CC=N1)C1=CN=C2SCCN12,s_22____14476978____13456062,0.999910,True
113986,Enamine_REAL_LeadLike_063,7092259,50.0,B_pockets,7092259,CC1=CC(F)=NC=C1NC(=O)C1=C2NC(O)=NC2=CN=C1,s_22____20075852____16803900,0.999915,True
113987,Enamine_REAL_LeadLike_088,8496329,50.0,B_pockets,8496329,O=C(CCC1=CN2N=NN=C2N=C1)OCC1COC(=O)N1,s_276436____14082488____31174282,0.999935,False


In [64]:
df[df['id'].str.contains("23521810")][:50]

,split,index,n_targets,label,index.1,smiles,id,rd,keep
6,Enamine_REAL_LeadLike_053,8627425,21.0,A_proteins,8627425,CC=CC(=O)N1CC=CC1C(=O)N1CCCC(O)CC1,m_274552____23521810____13186034____15986190,0.000160,True
877,Enamine_REAL_LeadLike_084,9589963,21.0,A_proteins,9589963,O=C(NC1=CC=C(O)C=C1)C1C=CCN1C(=O)C1CCO1,m_274552____23521810____24995232____15985840,0.033208,True
889,Enamine_REAL_LeadLike_084,9538074,21.0,A_proteins,9538074,NC(=O)C(=O)N1CC=CC1C(=O)NC1COCCC1(F)F,m_274552____23521810____15973642____15980482,0.033683,True
954,Enamine_REAL_LeadLike_053,8642655,21.0,A_proteins,8642655,CC1(O)CN(C(=O)C2C=CCN2C(=O)C2=NOC=C2)C1,m_274552____23521810____15958068____13454072,0.035462,True
1492,Enamine_REAL_LeadLike_084,9414478,21.0,A_proteins,9414478,O=C(NCC1=COC=N1)C1C=CCN1C(=O)C1CC=CC1,m_274552____23521810____12398572____13454210,0.055504,True
1952,Enamine_REAL_LeadLike_053,8657994,21.0,A_proteins,8657994,O=C(C1C=CCN1C(=O)C1=C(Cl)CC1)N1CC(CF)C1,m_274552____23521810____15968732____15986982,0.073293,True
2108,Enamine_REAL_LeadLike_084,9560735,21.0,A_proteins,9560735,[N-]=[N+]=NCCCC(=O)N1CC=CC1C(=O)N1CC=CCO1,m_274552____23521810____24976466____15975722,0.078416,True
2253,Enamine_REAL_LeadLike_053,8629868,21.0,A_proteins,8629868,[N-]=[N+]=NCC(=O)N1CC=CC1C(=O)NCC1=NN=CN1,m_274552____23521810____13730534____15979422,0.083175,True
2486,Enamine_REAL_LeadLike_084,9570803,21.0,A_proteins,9570803,CC=C1CN(C(=O)C2C=CCN2C(=O)[C@H]2CC=CCC2)C1,m_274552____23521810____24981984____25038678,0.092701,True
3256,Enamine_REAL_LeadLike_053,8628795,21.0,A_proteins,8628795,NC(=O)CNC(=O)C1C=CCN1C(=O)[C@@H]1COC(=O)N1,m_274552____23521810____13454578____15976290,0.122115,True


In [62]:
SYNTHON_COUNTS

defaultdict(int,
            {'30583064': 10,
             '18521434': 1,
             '17554698': 10,
             '14140274': 4,
             '19055260': 1,
             '29034160': 3,
             '1019008': 5,
             '13618256': 1,
             '9018314': 5,
             '7399710': 2,
             '20276634': 10,
             '903382': 1,
             '13259644': 8,
             '23521810': 10,
             '13186034': 1,
             '15986190': 10,
             '21917024': 1,
             '22359982': 10,
             '21950962': 3,
             '23821668': 1,
             '14587542': 4,
             '29025332': 2,
             '29066894': 2,
             '29315392': 1,
             '5635096': 5,
             '23075216': 4,
             '26278352': 4,
             '8805270': 10,
             '7404658': 2,
             '14484024': 6,
             '16783582': 2,
             '15735482': 2,
             '14780374': 1,
             '21554958': 2,
             '13970086': 5,
    